모델 배포 개론 08  
Last modified : 2026.03   
작성 : 박광성 (모두의연구소)  
수정 : 김지성 박기웅 (모두의연구소)  

In [1]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있다 — 최대 5분 대기 (첫 실행은 다운로드 포함)
    for i in range(600):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


# Day 8 — 자율 프로젝트: 나만의 모델 서빙 서비스 만들기

---

> **오늘의 목표**
>
> Day 1~7에서 배운 기술을 조합하여, 본인이 관심 있는 도메인의 모델을 서빙하는 서비스를 직접 만듭니다.  
> 따라하기가 아닌, **스스로 설계하고 구현하는** 첫 경험입니다.

---



## 1. 프로젝트 요구사항

---

### 1.1 조건

Day 5(주택 가격 예측)와 Day 6~7(이미지 분류 / 챗봇)에서 만든 서비스와 **동일한 구조**를 기본 베이스로 합니다.

```
필수 구현 항목:

1. FastAPI 백엔드
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)

2. API Key 인증
   - Day 6의 auth.py 재사용

3. Streamlit 프론트엔드
   - 사용자 입력 → API 호출 → 결과 표시

4. 에러 처리
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환
```



### 1.2 제한 사항

```
하지 않는 것:

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동
```



### 1.3 평가 기준

```
✅ 서버가 정상적으로 실행되는가?
✅ Swagger UI에서 추론이 동작하는가?
✅ API Key 없이 요청하면 401이 반환되는가?
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?
```

---

## 2. 모델 선택 가이드

---

### 2.1 Hugging Face에서 모델 찾기

[Hugging Face Models](https://huggingface.co/models)에서 사전학습 모델을 선택합니다.
모델 학습은 하지 않고, `from_pretrained()`으로 바로 사용할 수 있는 모델을 고릅니다.

**모델 선택 시 확인할 것:**

```
1. 태스크가 명확한가? (text-classification, image-classification, summarization 등)
2. 한국어를 지원하는가? (필수는 아니지만, 데모가 직관적입니다)
3. 모델 크기가 적당한가? (CPU 환경이면 500MB 이하를 권장합니다)
4. pipeline()으로 바로 사용 가능한가?
```



### 2.2 도메인별 추천 예시

아래는 예시일 뿐입니다. **본인이 관심 있는 도메인을 자유롭게 선택하세요.**

| 도메인 | 태스크 | 추천 모델 (예시) |
|---|---|---|
| 감정 분석 | `text-classification` | `snunlp/KR-FinBert-SC` |
| 뉴스 분류 | `text-classification` | 원하는 분류 모델 |
| 텍스트 요약 | `summarization` | `eenzeenee/t5-base-korean` |
| 번역 | `translation` | `Helsinki-NLP` 시리즈 |
| 이미지 분류 | `image-classification` | `google/vit-base-patch16` |
| 객체 탐지 | `object-detection` | `facebook/detr-resnet-50` |
| 질의 응답 | `question-answering` | 원하는 QA 모델 |



### 2.3 모델 동작 확인

모델을 선택했으면, **서버 코드를 작성하기 전에** 노트북에서 먼저 동작을 확인합니다.

In [2]:
# transformers 가 없으면 설치합니다. (Colab은 세션마다 확인이 필요합니다)
import importlib.util, sys, subprocess

for _pkg in ("transformers", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"❌ {_pkg} 미설치 → 지금 설치합니다.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

# ─────────────────────────────────────────────────────────────
# 도메인: 시니어 안심 서비스 "이거 봐줘"
#   부모님이 받은 문자/고지서/안내문을 6가지 종류로 분류한다.
#   (실제 서비스에서 쓰는 분류 체계를 그대로 사용)
#
# 모델 선택: zero-shot-classification (NLI 기반)
#   - 학습 금지 제약 하에서, 내 도메인 라벨(납부/사기 등)을 그대로 쓰려면
#     기성 감정분류 모델로는 불가능하다. (라벨 축이 다름)
#   - zero-shot NLI 모델은 candidate_labels로 내 분류 체계를 주입하면
#     추가 학습 없이 그대로 동작한다. → "왜 이 모델인가"가 가장 분명해지는 선택.
#   - MoritzLaurer/mDeBERTa-v3-base-xnli... : 한국어 포함 다국어 NLI, CPU 로드 가능.
# ─────────────────────────────────────────────────────────────
from transformers import pipeline

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

# 시니어 AI "이거 봐줘"의 분류 체계 (6종)
CANDIDATE_LABELS = [
    "납부 안내",       # 요금/세금/임대료 등 낼 돈이 있는 안내
    "행사 안내",       # 모임/행사/일정 안내
    "개인정보 안내",   # 비밀번호/보안/개인정보 관련 안내
    "중요한 공지",     # 공공기관/기업의 중요 공지
    "광고",           # 홍보/판촉/분양 등 광고
    "사기 의심",       # 피싱/스미싱/사기로 의심되는 내용
]

classifier = pipeline("zero-shot-classification", model=MODEL_NAME)

sample = "관리비 87,500원이 이번 달 25일까지 미납되었습니다. 기한 내 납부 부탁드립니다."
result = classifier(sample, candidate_labels=CANDIDATE_LABELS)
print("입력:", sample)
print("→ 예측:", result["labels"][0], f"(score={result['scores'][0]:.3f})")
print("전체 순위:", list(zip(result["labels"], [round(s,3) for s in result["scores"]])))


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

입력: 관리비 87,500원이 이번 달 25일까지 미납되었습니다. 기한 내 납부 부탁드립니다.
→ 예측: 납부 안내 (score=0.595)
전체 순위: [('납부 안내', 0.595), ('광고', 0.147), ('개인정보 안내', 0.092), ('중요한 공지', 0.075), ('행사 안내', 0.068), ('사기 의심', 0.024)]


예시 코드입니다. 샘플 이미지 파일이 있어야 실행됩니다.

```python
# 예시: 이미지 분류
from transformers import pipeline

classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

result = classifier("test_image.jpg")
print(result)
# [{'label': 'tabby cat', 'score': 0.82}, ...]
```

> **이 셀이 정상 동작하면**, 서버 코드 작성으로 넘어갑니다.
> 여기서 에러가 나면, 모델을 바꾸거나 입력 형식을 확인하세요.



---

## 3. 프로젝트 뼈대 코드

---



### 3.1 폴더 구조

In [3]:
import os

dirs = ["app", "models", "frontend"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt
""")

프로젝트 구조:

my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt



### 3.2 auth.py — 재사용

In [4]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Writing app/auth.py


### 3.3 schemas.py — 직접 작성

In [5]:
%%writefile app/schemas.py
"""
입력/출력 스키마 정의 — 시니어 AI "이거 봐줘" 문자 분류

입력:  분류할 문자/고지서 텍스트 한 개
출력:  예측 라벨 + 신뢰도 + (부모님용) 배지 문구/색
참고:  Day 2 섹션 4 (Pydantic), Day 5 섹션 3 (HousingRequest)
"""
from typing import List
from pydantic import BaseModel, Field


class PredictRequest(BaseModel):
    # 필수 필드: 분류할 텍스트. 빈 문자열/과도한 길이를 검증으로 막는다.
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분류할 문자/고지서 내용",
        examples=["관리비 87,500원이 이번 달 25일까지 미납되었습니다."],
    )


class LabelScore(BaseModel):
    label: str
    score: float


class PredictResponse(BaseModel):
    success: bool = True
    label: str                    # 최종 예측 종류 (예: "납부 안내")
    confidence: float             # 최종 라벨의 신뢰도 (0~1)
    badge_text: str               # 부모님용 안내 문구 (예: "지금 확인하세요")
    badge_color: str              # green / yellow / gray / red
    ranking: List[LabelScore]     # 전체 라벨별 점수 (설명/디버깅용)


Writing app/schemas.py


### 3.4 model_service.py — 직접 작성

In [6]:
%%writefile app/model_service.py
"""
모델 로드 + 추론 — zero-shot 분류로 "이거 봐줘"의 6종 분류를 수행

참고: Day 1 섹션 5 (모델 로드), Day 5 섹션 2 (Predictor)

핵심: 학습 없이 내 분류 체계를 쓰기 위해 zero-shot-classification 사용.
      candidate_labels 로 6종 라벨을 주입하고, 결과 라벨에 따라
      부모님용 배지 문구/색을 규칙으로 매핑한다. (서비스 로직 재현)
"""
from transformers import pipeline

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

# 시니어 AI "이거 봐줘"의 분류 체계 (6종)
CANDIDATE_LABELS = [
    "납부 안내",
    "행사 안내",
    "개인정보 안내",
    "중요한 공지",
    "광고",
    "사기 의심",
]

# 종류 → (배지 문구, 색). 서비스에서 쓰는 매핑을 그대로 옮김.
BADGE_MAP = {
    "납부 안내":     ("지금 확인하세요", "green"),
    "개인정보 안내": ("지금 확인하세요", "green"),
    "중요한 공지":   ("급하지 않아요",   "yellow"),
    "행사 안내":     ("급하지 않아요",   "yellow"),
    "광고":         ("무시하셔도 돼요", "gray"),
    "사기 의심":     ("주의하세요",     "red"),
}


def load_model():
    """zero-shot 분류 파이프라인을 로드하여 반환한다."""
    return pipeline("zero-shot-classification", model=MODEL_NAME)


def predict(model, text: str) -> dict:
    """텍스트 1건을 6종으로 분류하고, 배지 문구/색까지 붙여 반환한다."""
    output = model(text, candidate_labels=CANDIDATE_LABELS)

    label = output["labels"][0]
    confidence = float(output["scores"][0])
    badge_text, badge_color = BADGE_MAP.get(label, ("확인이 필요해요", "gray"))

    ranking = [
        {"label": l, "score": round(float(s), 4)}
        for l, s in zip(output["labels"], output["scores"])
    ]

    return {
        "label": label,
        "confidence": round(confidence, 4),
        "badge_text": badge_text,
        "badge_color": badge_color,
        "ranking": ranking,
    }


Writing app/model_service.py


### 3.5 main.py — 직접 작성

In [7]:
%%writefile app/main.py
"""
FastAPI 서버 — 시니어 AI "이거 봐줘" 문자 분류 API

참고: Day 5 섹션 3 (housing_api.py), Day 6 섹션 6 (image_api.py)

구성:
  - startup 에서 모델 1회 로드 (요청마다 로드하지 않도록)
  - GET  /health   : 상태 확인
  - POST /predict  : 인증 + 입력검증 + 비동기 추론
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict

app = FastAPI(title="이거 봐줘 — 문자 분류 API", version="1.0")

# 모델과 스레드풀은 앱 전역에서 1개만 유지
_state = {"model": None}
_executor = ThreadPoolExecutor(max_workers=2)


@app.on_event("startup")
def _startup():
    # 서버 기동 시 모델을 한 번만 로드한다. (첫 로드는 시간이 걸릴 수 있음)
    print("모델 로딩 중...")
    _state["model"] = load_model()
    print("모델 로드 완료")


@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": _state["model"] is not None}


@app.post("/predict", response_model=PredictResponse)
async def predict_endpoint(
    req: PredictRequest,
    user: str = Depends(verify_api_key),   # API Key 없으면 여기서 401
):
    model = _state["model"]
    if model is None:
        # 모델이 아직 로드되지 않았으면 503
        raise HTTPException(status_code=503, detail="모델이 아직 준비되지 않았습니다.")

    try:
        # CPU 추론은 블로킹이므로 스레드풀에서 실행해 이벤트 루프를 막지 않는다.
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(_executor, predict, model, req.text)
    except Exception as e:
        # 모델 추론 중 예외 → 500
        raise HTTPException(status_code=500, detail=f"추론 중 오류: {e}")

    return PredictResponse(success=True, **result)


Writing app/main.py


### 3.6 frontend/app.py — 직접 작성

In [8]:
%%writefile frontend/app.py
"""
Streamlit 프론트엔드 — "이거 봐줘" 데모 UI

참고: Day 4 섹션 6 (대시보드), Day 5 섹션 4 (주택 가격 UI)

흐름: API Key 입력 → 문자 붙여넣기 → [분석] → 배지 + 종류 + 신뢰도 표시
"""
import streamlit as st
import requests

API_URL = "http://localhost:8000/predict"

# 배지 색 → 이모지 동그라미 (부모님용 신호등)
COLOR_DOT = {"green": "🟢", "yellow": "🟡", "gray": "⚪", "red": "🔴"}

st.set_page_config(page_title="이거 봐줘", page_icon="📩")
st.title("📩 이거 봐줘 — 문자 분류")
st.caption("부모님이 받은 문자·고지서가 무슨 내용인지 알려드려요.")

# 사이드바: API Key
with st.sidebar:
    st.header("설정")
    api_key = st.text_input("API Key", value="test-key-001", type="password")
    st.caption("데모 키: test-key-001 / test-key-002")

# 입력
text = st.text_area(
    "받은 문자 내용을 붙여넣어 주세요",
    height=160,
    placeholder="여기에 문자 내용을 붙여넣어 주세요.",
)

if st.button("분석하기", type="primary"):
    if not text.strip():
        st.warning("문자 내용을 입력해 주세요.")
    else:
        try:
            resp = requests.post(
                API_URL,
                json={"text": text},
                headers={"X-API-Key": api_key},
                timeout=60,
            )
        except requests.exceptions.RequestException as e:
            st.error(f"서버에 연결할 수 없습니다: {e}")
        else:
            if resp.status_code == 401:
                st.error("API Key가 올바르지 않습니다. (401)")
            elif resp.status_code == 422:
                st.error("입력이 올바르지 않습니다. (422)")
            elif resp.status_code != 200:
                st.error(f"오류가 발생했습니다. ({resp.status_code}) {resp.text}")
            else:
                data = resp.json()
                dot = COLOR_DOT.get(data["badge_color"], "⚪")
                st.markdown(f"## {dot} {data['badge_text']}")
                st.markdown(f"**종류:** {data['label']}  \n**신뢰도:** {data['confidence']:.1%}")

                with st.expander("자세히 보기 (전체 분류 점수)"):
                    for item in data["ranking"]:
                        st.write(f"- {item['label']}: {item['score']:.1%}")


Writing frontend/app.py


---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

> ⚠️ **코드를 수정했는데 반영이 안 될 때 — 커널을 재시작하세요.**
>
> `app/main.py`, `app/model_service.py` 등을 고친 뒤 아래 셀을 다시 실행해도,
> 이미 메모리에 올라간 이전 코드가 남아 변경이 반영되지 않을 수 있습니다.
> 코드를 수정했다면 **커널 재시작(Kernel → Restart) 후 맨 위 셀부터 다시 실행**하세요.

In [9]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
# app/main.py 를 먼저 불러와 보고, 문제가 있으면 원인을 그대로 보여줍니다.
import importlib, sys, traceback

sys.modules.pop("app.main", None)          # 파일을 고친 경우 최신 내용 반영
ready = False
try:
    _main = importlib.import_module("app.main")
    if hasattr(_main, "app"):
        ready = True
    else:
        print("❌ app/main.py 에 FastAPI 객체 `app` 이 없습니다 — 3.5를 먼저 완성하세요.")
except Exception:
    print("❌ app/main.py 를 불러올 수 없습니다 — 아래 오류를 확인하세요:")
    traceback.print_exc()

if ready:
    serve_in_thread("app.main:app", port=8000)


모델 로딩 중...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


#### Swagger UI 열기

서버가 떴으면 Swagger UI에서 API를 직접 호출해 볼 수 있습니다.  

- 로컬: 브라우저에서 http://localhost:8000/docs
- Colab: localhost 접속이 안 되므로, 아래 셀이 노트북 안에 Swagger UI를 띄웁니다.

> 발표(5.1)의 데모 시연에 이 화면을 그대로 쓸 수 있습니다.


In [10]:
# Swagger UI를 노트북 안에서 열기
try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import output
    print("✅ Swagger UI를 아래에 띄웁니다 (Colab 프록시 → 포트 8000)")
    output.serve_kernel_port_as_iframe(8000, path="/docs", height="800")
else:
    from IPython.display import IFrame, display
    print("✅ Swagger UI를 아래에 띄웁니다: http://localhost:8000/docs")
    display(IFrame("http://localhost:8000/docs", width="100%", height=800))


✅ Swagger UI를 아래에 띄웁니다 (Colab 프록시 → 포트 8000)


<IPython.core.display.Javascript object>

### 4.3 API 테스트 템플릿 (Step 5에서 사용)

In [11]:
import requests

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

# health check
print(requests.get(f"{API_URL}/health").json())

# 추론 테스트 — 본인의 입력에 맞게 수정하세요
response = requests.post(
    f"{API_URL}/predict",
    json={"여기에": "본인의 입력"},   # ← 수정
    headers=HEADERS,
)
print(f"상태: {response.status_code}")
print(f"결과: {response.json()}")

{'status': 'ok', 'model_loaded': True}
상태: 422
결과: {'detail': [{'type': 'missing', 'loc': ['body', 'text'], 'msg': 'Field required', 'input': {'여기에': '본인의 입력'}}]}


### 4.4 프론트엔드 실행 (Step 6에서 사용)

`frontend/app.py`를 작성한 뒤 아래 셀로 띄웁니다.  
노트북 셀에서 `streamlit run`을 직접 실행하면 셀이 끝나지 않으니, 백그라운드로 실행합니다.


In [12]:
# Colab은 세션이 새로 뜰 때마다 streamlit 설치가 필요합니다.
try:
    import streamlit
except ImportError:
    print("❌ Streamlit 미설치 → 지금 설치합니다. (1분 정도 걸립니다)")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)

import sys, subprocess, time, socket, contextlib, tempfile, os

try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run_streamlit(script, port=8501):
    """Streamlit을 백그라운드로 띄우고 '실제로 떴는지'까지 확인한다. (Windows/macOS/Linux 공통)"""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if port_open(port):                      # 이미 떠 있으면 재사용
        print(f"♻️  이미 실행 중 (포트 {port})")
        return None

    # 로그는 파일로 — 파이프가 가득 차 서버가 멈추는 일을 막고, 실패 시 원인을 읽을 수 있다
    log_path = os.path.join(tempfile.gettempdir(), f"streamlit_{port}.log")
    log = open(log_path, "w", encoding="utf-8")

    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", script,
         "--server.port", str(port),
         "--server.enableCORS", "false",           # iframe은 '다른 주소'로 취급됩니다 —
         "--server.enableXsrfProtection", "false", #   끄지 않으면 화면이 계속 로딩만 됩니다
         "--server.headless", "true"],             # 최초 실행 '이메일 프롬프트'를 건너뜀
        stdout=log, stderr=subprocess.STDOUT,      #   (없으면 입력을 기다리다 조용히 죽음)
    )
    for _ in range(60):                      # 최대 15초, 0.25초 간격 확인
        if proc.poll() is not None:          # 일찍 죽음 → 로그를 보여줌
            log.close()
            print(f"❌ Streamlit이 종료됨 (code {proc.returncode}) — 로그:")
            print(open(log_path, encoding="utf-8").read()[-2000:])
            return proc
        if port_open(port):
            if IN_COLAB:                     # Colab의 localhost는 '내 PC'가 아니라 Colab 서버입니다
                print(f"✅ 프론트엔드 실행됨 (포트 {port}) — 아래 Step 3에서 화면을 띄웁니다")
            else:
                print(f"✅ 프론트엔드: http://localhost:{port}")
            print(f"   (로그: {log_path})")
            return proc
        time.sleep(0.25)
    proc.terminate(); log.close()
    print(f"❌ 15초 내에 포트가 열리지 않음 — 로그:")
    print(open(log_path, encoding="utf-8").read()[-2000:])
    return proc

proc = run_streamlit("frontend/app.py", port=8501)


❌ Streamlit 미설치 → 지금 설치합니다. (1분 정도 걸립니다)
✅ 프론트엔드 실행됨 (포트 8501) — 아래 Step 3에서 화면을 띄웁니다
   (로그: /tmp/streamlit_8501.log)


#### 노트북에서 바로 확인

> ⚠️ **화면이 비어 있거나 계속 로딩만 된다면**
> - 위 셀이 `✅`를 출력했는지 먼저 확인하세요.
> - Colab은 런타임에 연결된 상태에서만 iframe을 그립니다. 셀을 다시 실행해보세요.
> - `frontend/app.py`가 비어 있으면 빈 화면이 나옵니다. 3.6을 먼저 완성하세요.


In [13]:
def show_dashboard(port=8501, height=900):
    """실행 중인 프론트엔드를 노트북 셀 안에 iframe으로 띄웁니다."""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if not port_open(port):                  # Step 2를 건너뛴 경우
        print(f"⚠️ 포트 {port}에 프론트엔드가 없습니다. Step 2 셀을 먼저 실행하세요.")
        return

    if IN_COLAB:
        from google.colab import output
        print(f"✅ 대시보드를 아래에 띄웁니다 (Colab 프록시 → 포트 {port})")
        output.serve_kernel_port_as_iframe(port, height=str(height))
    else:
        from IPython.display import IFrame, display
        print(f"✅ 대시보드를 아래에 띄웁니다: http://localhost:{port}")
        display(IFrame(f"http://localhost:{port}", width="100%", height=height))

show_dashboard(port=8501, height=900)


✅ 대시보드를 아래에 띄웁니다 (Colab 프록시 → 포트 8501)


<IPython.core.display.Javascript object>

### 4.5 막혔을 때 참고할 Day



```
"스키마를 어떻게 정의하지?"        → Day 2 섹션 4, Day 5 섹션 3
"FastAPI 서버 구조가 기억 안 나"   → Day 5 섹션 3, Day 6 섹션 6
"run_in_executor 사용법?"         → Day 3 섹션 4
"인증 적용 방법?"                  → Day 6 섹션 2
"Streamlit에서 API 호출?"         → Day 4 섹션 6, Day 5 섹션 4
"에러 처리?"                      → Day 3 섹션 5
```

---

## 5. 발표 및 회고

---

### 5.1 발표 (개인당 5분)

```
발표 항목:
  1. 어떤 도메인/태스크를 선택했는가?
  2. 어떤 모델을 사용했는가? (선택 이유)
  3. 데모 시연 (Swagger UI 또는 Streamlit)
  4. 구현하면서 가장 어려웠던 부분은?
```

### 5.2 회고

```
스스로 돌아보기:
  - Day 1~7 교안 없이 코드를 작성할 수 있었는가?
  - 어떤 부분에서 교안을 다시 찾아봤는가?
  - 다음에 다시 만든다면 무엇을 다르게 하겠는가?
```

---

## 6. 8일간의 여정 정리

---



### 6.1 Day 1의 문제 → Day 8의 해결

```
Day 1의 문제                           해결한 Day
──────────────────────────────        ──────────
라이브러리가 없음                       Day 1: requirements.txt
모델 구조 코드 필요                     Day 1: model_utils.py 모듈 분리
전처리 로직 누락                       Day 5/7: 전처리 파라미터 저장
비개발자가 사용할 수 없음               Day 4/5/7: Streamlit UI
누구나 API 호출 가능                   Day 6: API Key 인증
스스로 서비스를 만들 수 있는가?          Day 8: 자율 프로젝트 ✅
```



### 6.2 8일간 배운 기술 전체 지도

```
Day 1: 환경 세팅 + 모델 직렬화          "모델을 저장하고 불러온다"
Day 2: FastAPI + Pydantic              "모델을 API로 감싼다"
Day 3: 비동기 처리 + 에러/로깅          "안정적으로 돌아가게 한다"
Day 4: Streamlit + 시스템 아키텍처      "누구나 쓸 수 있게 한다"
Day 5: [프로젝트 1] 정형 데이터 서비스   "따라하며 조립한다"
Day 6: 인증 + 파일 업로드               "보안과 비정형 데이터를 다룬다"
Day 7: [프로젝트 2] 텍스트/이미지 서비스  "패턴을 반복하며 익힌다"
Day 8: [자율 프로젝트] 나만의 서비스      "스스로 만든다"
```



### 6.3 Next Step: MLOps로 가는 길

```
이 과정에서 배운 것:                 다음 과정에서 배울 것:
──────────────────                  ──────────────────
수동으로 서버 실행                    → Docker로 패키징
단일 서버에서 실행                    → 클라우드 배포 (AWS, GCP)
코드 변경 시 수동 재시작              → CI/CD 파이프라인 (자동 빌드/배포)
모델 버전 1개                        → 모델 버전 관리 (MLflow, DVC)
수동 모니터링 (로그 확인)             → 자동 모니터링 (Prometheus, Grafana)
```



> **"코드를 고칠 때마다 매번 서버를 재시작해야 하나요?"**
>
> 그 질문의 답이 MLOps입니다.
> CI/CD 파이프라인이 코드 변경을 감지하면 자동으로 빌드, 테스트, 배포합니다.
> 여러분은 코드를 커밋하기만 하면 됩니다.

---



### ✅ Day 8 최종 체크포인트

```
Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?
Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?
Q3. run_in_executor를 사용한 이유는 무엇입니까?
Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?
Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?
```

---

### 📌 Day 8 요약 & 전체 과정 완료

```
오늘 한 일:
  ✅ Day 1~7의 기술을 조합하여 나만의 서비스를 직접 만들었습니다.
  ✅ 교안 없이 설계 → 구현 → 테스트를 경험했습니다.
  ✅ 8일간의 여정을 회고하고, MLOps로 가는 길을 확인했습니다.

8일간의 전체 성과:
  🎉 PyTorch / HuggingFace 모델을 API로 서빙할 수 있습니다.
  🎉 비개발자도 사용 가능한 웹 UI를 붙일 수 있습니다.
  🎉 인증, 에러 처리, 로깅으로 안정적인 서비스를 만들 수 있습니다.
  🎉 스스로 설계하고 구현할 수 있다는 자신감을 얻었습니다.
```

### 제출

06DP08

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다  

1. 프로젝트 실행 내역 캡쳐와 설명
2. 각 섹션 체크포인트의 답변
3. 프로젝트 회고

수고하셨습니다!